### `list`, `tuple`, `dict`, `set`

In [ ]:
simple_list = [1, 2, 3] # list of ints
simple_tuple = (1, 2, 3) # tuple of ints
simple_dict = {"a": 1, "b": 2} # simple dict
simple_set = {0, 1, 2} # simple set

In [ ]:
import time

n = 1_000_000
data_list = list(range(n))
data_set = set(range(n))

target = n - 1

start = time.perf_counter()
target in data_list
print(f"list: {time.perf_counter() - start:.6f} сек")

start = time.perf_counter()
target in data_set
print(f"set: {time.perf_counter() - start:.6f} сек")

#### hint
Под капотом `dict` и `set` - это хеш-таблицы: значение находится по хешу ключа/элемента напрямую, без перебора. `list` требует линейного просмотра, если не известна позиция.

In [ ]:
coords = {(55.75, 37.61): "Москва"}

try:
    bad = {[1, 2]: "значение"}
except TypeError as e:
    print(e) 

print(hash((1, 2, "a")))
print(hash(frozenset([1, 2, 3])))

#### hint
Ключом словаря или элементом множества может быть только **неизменяемый и хешируемый** объект.


In [ ]:
email = "smth@gm.com"

blacklist = ["spam@gm.com", "income@gm.com", "smth@gm.com"]
if email in blacklist:   # O(n) на каждой проверке
    pass

blacklist = {"spam@gm.com", "income@gm.com", "smth@gm.com"}
if email in blacklist:   # O(1)
    pass

#### `is` vs `==`

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b)   # True  - значения равны
print(a is b)   # False - разные объекты в памяти
print(a is c)   # True  - это один и тот же объект

print(id(a), id(b), id(c))

In [ ]:
value = None
if value is None:
    print('is None')
if value == None:
    print('== None')

In [ ]:
import numpy as np

arr = np.array([1, 2, 3])

print(arr == None)
print(arr is None)

### Изменяемые (mutable) и неизменяемые (immutable) типы

In [ ]:
mutable_list = [1, 2, 3]
mutable_list[2] = 4
print(mutable_list)

immutable_tuple = (1, 2, 3)
immutable_tuple[0] = 1

In [ ]:
def add_item(item, container=[]):
    container.append(item)
    return container

print(add_item(1))
print(add_item(2))

### Поверхностное и глубокое копирование

In [ ]:
import copy

original = {"name": "Заказ 1", "items": [1, 2, 3]}

shallow = original.copy()
deep = copy.deepcopy(original)

shallow["items"].append(4)
print(original["items"])  # [1, 2, 3, 4] - вложенный список общий

deep["items"].append(5)
print(original["items"])  # [1, 2, 3, 4] - deep полностью независим

### Типичные ошибки с алиасингом

In [ ]:
# Ошибка: "размножение" вложенных списков через умножение
grid = [[0] * 3] * 3   # все три строки - ОДИН И ТОТ ЖЕ объект list!
grid[0][0] = 1
print(grid)   # [[1, 0, 0], [1, 0, 0], [1, 0, 0]] - испортились все строки

# Правильно:
grid = [[0] * 3 for _ in range(3)]   # каждая строка - новый список
grid[0][0] = 1
print(grid)   # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [ ]:
# Передача изменяемого объекта в функцию - функция может изменить оригинал
def add_tax(order):
    order["total"] *= 1.2   # изменяет исходный словарь!
    return order

order = {"total": 100}
add_tax(order)
print(order["total"])   # 120.0 - оригинал изменился, это может быть неожиданно

### `Counter` - подсчёт элементов


In [ ]:
from collections import Counter

words = ["python", "java", "c++", "fortran", "python", "c++"]

counts = Counter(words)
print(counts)                    # Counter({'python': 2, 'c++': 2, 'java': 1, 'fortran': 1})
print(counts.most_common(2))     # [('python', 2), ('c++', 2)]
print(counts["assembler"])       # 0 - не бросает KeyError, в отличие от обычного dict!

# Арифметика счётчиков
c1 = Counter(["a", "b", "a"])
c2 = Counter(["a", "c"])
print(c1 + c2)   # Counter({'a': 3, 'b': 1, 'c': 1})
print(c1 - c2)   # Counter({'a': 1, 'b': 1})

### `defaultdict` - словарь со значением по умолчанию

In [ ]:
from collections import defaultdict

# Без defaultdict - приходится проверять наличие ключа вручную
totals = {}
for category, amount in [("еда", 100), ("еда", 50), ("транспорт", 30)]:
    if category not in totals:
        totals[category] = 0
    totals[category] += amount

# С defaultdict - короче и без ветвлений
totals = defaultdict(int)
for category, amount in [("еда", 100), ("еда", 50), ("транспорт", 30)]:
    totals[category] += amount   # ключ создаётся автоматически со значением int() == 0

print(dict(totals))   # {'еда': 150, 'транспорт': 30}

# defaultdict(list) - группировка элементов
groups = defaultdict(list)
for category, order_id in [("еда", 1), ("еда", 2), ("транспорт", 3)]:
    groups[category].append(order_id)

print(dict(groups))   # {'еда': [1, 2], 'транспорт': [3]}

### `namedtuple` - легковесные структуры с именованными полями

In [ ]:
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)

print(p.x, p.y)      # 3 4 - обращение по имени, а не только по индексу
print(p[0], p[1])    # 3 4 - но и по индексу тоже работает
print(p)             # Point(x=3, y=4)

# Неизменяемость, как у обычного tuple
try:
    p.x = 10
except AttributeError as e:
    print(e)   # невозможно присвоить аттрибуту значение

### `OrderedDict`

In [ ]:
from collections import OrderedDict

# С Python 3.7+ обычный dict уже сохраняет порядок вставки,
# но OrderedDict остаётся полезен для явного сравнения порядка
# и метода move_to_end()

od = OrderedDict()
od["Ulysses"] = 1
od["The Human Comedy"] = 2
od["Animal Farm"] = 3

od.move_to_end("Ulysses")
print(od)   # OrderedDict({'The Human Comedy': 2, 'Animal Farm': 3, 'Ulysses': 1})

# Частый кейс: LRU-кеш вручную (обратный LRU)
od.move_to_end("Animal Farm", last=False)   # переместить в начало
print(od)

### List, dict и set comprehensions

In [ ]:
squares = [x ** 2 for x in range(10)]

square_map = {x: x ** 2 for x in range(10)}

unique_lengths = {len(word) for word in ["кот", "собака", "кит", "слон"]}

# С условием
even_squares = [x ** 2 for x in range(10) if x % 2 == 0]

# Вложенный comprehension - "выпрямление" матрицы
matrix = [[1, 2, 3], [4, 5, 6]]
flat = [x for row in matrix for x in row]   # [1, 2, 3, 4, 5, 6]

### Производительность: comprehension vs цикл vs `map`/`filter`


In [ ]:
import timeit

n = 1_000_000

t_loop = timeit.timeit(
    "result = []\nfor x in range(n): result.append(x * 2)",
    globals={"n": n}, number=5
)

t_comp = timeit.timeit(
    "[x * 2 for x in range(n)]",
    globals={"n": n}, number=5
)

t_map = timeit.timeit(
    "list(map(lambda x: x * 2, range(n)))",
    globals={"n": n}, number=5
)

print(f"цикл: {t_loop:.3f}, comprehension: {t_comp:.3f}, map: {t_map:.3f}")
# comprehension обычно быстрее ручного цикла за счёт оптимизации на уровне байткода

In [ ]:
import dis

def loop_version(n):
    result = []
    for x in range(n):
        result.append(x * 2)### 4.3 Обработка JSON-подобных данных

    return result

def comp_version(n):
    return [x * 2 for x in range(n)]

print("=== LOOP BYTECODE ===")
dis.dis(loop_version)
print("\n=== COMPREHENSION BYTECODE ===")
dis.dis(comp_version)

### Обработка JSON-подобных данных

In [ ]:
orders = [
    {"id": 1, "customer": {"name": "Иван", "city": "Москва"}, "items": [{"item": "book", "qty": 2}]},
    {"id": 2, "customer": {"name": "Борис", "city": "Казань"}, "items": [{"item": "book", "qty": 1}, {"item": "pen", "qty": 3}]},
]

# Извлечь все города
cities = {order["customer"]["city"] for order in orders}

# Посчитать общее количество по item - вложенный обход через collections
from collections import defaultdict

item_totals = defaultdict(int)
for order in orders:
    for item in order["items"]:
        item_totals[item["item"]] += item["qty"]

print(dict(item_totals))   # {'book': 5, 'pen': 1}

# Безопасное извлечение при отсутствующих полях - .get() вместо []
discount = order.get("discount", 0)   # не упадёт, если ключа нет
order['discount']

### 4.4 Рекурсивный обход вложенных структур

In [ ]:
def flatten(data):
    """Рекурсивно 'выпрямляет' произвольно вложенные списки."""
    result = []
    for item in data:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result

nested = [1, [2, 3, [4, 5], 6], [7, [8, [9]]]]
print(flatten(nested))   # [1, 2, 3, 4, 5, 6, 7, 8, 9]

def find_all_keys(data, target_key, path=""):
    """Рекурсивно ищет все вхождения ключа в произвольно вложенном dict/list."""
    found = []
    if isinstance(data, dict):
        for key, value in data.items():
            current_path = f"{path}.{key}" if path else key
            if key == target_key:
                found.append((current_path, value))
            found.extend(find_all_keys(value, target_key, current_path))
    elif isinstance(data, list):
        for i, item in enumerate(data):
            found.extend(find_all_keys(item, target_key, f"{path}[{i}]"))
    return found

print(find_all_keys(orders, "city"))
# [('[0].customer.city', 'Москва'), ('[1].customer.city', 'Казань')]

### Необработанные (сырые) данные

In [22]:
raw_orders = [
    {"category": "еда", "amount": 450.0},
    {"category": "транспорт", "amount": 120.0},
    {"category": "еда"},                               # отсутствует amount
    {"amount": 900.0},                                 # отсутствует category
    {"category": "еда", "amount": 300.0},
    {"category": "транспорт", "amount": 80.0},
    {"category": "развлечения", "amount": None},       # amount = None
]

#### Простой наивный способ

In [ ]:
totals = {}
counts = {}
for order in raw_orders:
    category = order.get("category")
    amount = order.get("amount")
    if category is None or amount is None:
        continue
    if category not in totals:
        totals[category] = 0.0
        counts[category] = 0
    totals[category] += amount
    counts[category] += 1

print(totals)
print(counts)

In [ ]:
from collections import defaultdict, Counter

totals = defaultdict(float)
counts = Counter()

for order in raw_orders:
    category = order.get("category")
    amount = order.get("amount")
    if category is None or amount is None:
        continue
    totals[category] += amount
    counts[category] += 1

print(dict(totals))    # {'еда': 750.0, 'транспорт': 200.0}
print(counts)          # Counter({'еда': 2, 'транспорт': 2})